In [ ]:
!pip show konlpy
!pip show wordcloud

Name: konlpy
Version: 0.6.0
Summary: Python package for Korean natural language processing.
Home-page: http://konlpy.org
Author: Team KoNLPy
Author-email: konlpy@googlegroups.com
License: GPL v3
Location: /usr/local/lib/python3.7/dist-packages
Requires: numpy, lxml, JPype1
Required-by: 
Name: wordcloud
Version: 1.5.0
Summary: A little word cloud generator
Home-page: https://github.com/amueller/word_cloud
Author: Andreas Mueller
Author-email: t3kcit+wordcloud@gmail.com
License: MIT
Location: /usr/local/lib/python3.7/dist-packages
Requires: numpy, pillow
Required-by: 


In [ ]:
%%bash
apt-get update
apt-get install g++ openjdk-8-jdk python-dev python3-dev
pip3 install JPype1
pip3 install konlpy

Hit:1 http://archive.ubuntu.com/ubuntu bionic InRelease
Get:2 http://archive.ubuntu.com/ubuntu bionic-updates InRelease [88.7 kB]
Hit:3 http://ppa.launchpad.net/c2d4u.team/c2d4u4.0+/ubuntu bionic InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64  InRelease
Get:5 http://security.ubuntu.com/ubuntu bionic-security InRelease [88.7 kB]
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu bionic-cran40/ InRelease
Hit:7 http://ppa.launchpad.net/cran/libgit2/ubuntu bionic InRelease
Get:8 http://archive.ubuntu.com/ubuntu bionic-backports InRelease [74.6 kB]
Hit:9 http://ppa.launchpad.net/deadsnakes/ppa/ubuntu bionic InRelease
Hit:10 http://ppa.launchpad.net/graphics-drivers/ppa/ubuntu bionic InRelease
Ign:11 https://developer.download.nvidia.com/compute/machine-learning/repos/ubuntu1804/x86_64  InRelease
Hit:12 https://developer.download.nvidia.com/compute/machine-learning/repos/ubuntu1804/x86_64  Release
Fetched 252 kB in 3s (96.4 kB/s)
Reading package li

In [ ]:
%env JAVA_HOME "/usr/lib/jvm/java-8-openjdk-amd64"

env: JAVA_HOME="/usr/lib/jvm/java-8-openjdk-amd64"


In [ ]:
from konlpy.tag import Okt

In [ ]:
from wordcloud import WordCloud, STOPWORDS

In [ ]:
import pandas as pd
import nltk
import numpy as np

In [ ]:
import re
from konlpy.tag import Okt
from nltk.corpus import stopwords
import nltk
import pandas as pd
import matplotlib.pyplot as plt
from gensim.models.word2vec import Word2Vec

In [ ]:
df = pd.read_excel('/content/감성분석반영리뷰_1_.xlsx')
df

,관광지,전체내용,리뷰내용_1,리뷰내용_0,긍정확률_0,부정확률_0,긍정확률_1,부정확률_1
0,경복궁,"['대한민국의 역사 ', ' 대한민국의 역사가 잠들어 있는 곳. 서울을 방문했다면 ...",대한민국의 역사가 잠들어 있는 곳. 서울을 방문했다면 꼭 방문해야 되는 곳. 경복...,대한민국의 역사,0.381490,0.618510,0.978027,0.021973
1,경복궁,"['국민이 공감하는 장소 ', ' 경복궁은 국민들이 자주 찾는곳으로 작성자는 주말에...",경복궁은 국민들이 자주 찾는곳으로 작성자는 주말에 자주 가족들과 방문하고 있음.특...,국민이 공감하는 장소,0.850068,0.149932,0.980540,0.019460
2,경복궁,"['산책하기 좋은 경복궁 ', ' 날씨 좋은 날 종종 산책하러 경복궁에 가는데 마음...",날씨 좋은 날 종종 산책하러 경복궁에 가는데 마음이 편온해지는 기분이라고 할까요?...,산책하기 좋은 경복궁,0.916744,0.083256,0.990634,0.009366
3,경복궁,"['Good ', ' Goooooood 다 좋습니다 다음에 또 오고 싶네요 근처 관...",Goooooood 다 좋습니다 다음에 또 오고 싶네요 근처 관광지도많고 먹을거리도...,Good,0.861854,0.138146,0.989062,0.010938
4,경복궁,"['가족단위로 방문하기 좋은곳 ', ' 요새 더더욱 코로나로 인해 사람 방문이 적음...",요새 더더욱 코로나로 인해 사람 방문이 적음. 두자녀 동반시 성인 입장무료. 지금...,가족단위로 방문하기 좋은곳,0.931031,0.068969,0.860709,0.139291
...,...,...,...,...,...,...,...,...
5351,순천 드라마세트장,['서울 달동네와 어린 시절에 보고 경험해 익숙한 셋트장에 잊혀진 기억들이 새록 새...,리뷰없음,서울 달동네와 어린 시절에 보고 경험해 익숙한 셋트장에 잊혀진 기억들이 새록 새록특...,0.311129,0.688871,0.005893,0.994107
5352,순천 드라마세트장,['사진찍기 좋은 날에~~^^'],리뷰없음,사진찍기 좋은 날에,0.848867,0.151133,0.005893,0.994107
5353,순천 드라마세트장,['기분 좋은 그 어떤 날~~~ 그 곳에선 그 어떠한 모습도 예쁘...,리뷰없음,기분 좋은 그 어떤 날 그 곳에선 그 어떠한 모습도 예쁘고 사랑...,0.962142,0.037858,0.005893,0.994107
5354,순천 드라마세트장,['재밌네요ㅎ'],리뷰없음,재밌네요ㅎ,0.934410,0.065590,0.005893,0.994107


In [ ]:
df = df.dropna()

In [ ]:
okt = Okt() 
noun_list = []
detokenized_doc = []

for sentence in df.리뷰내용_1:
    noun_list = okt.nouns(sentence)
    noun_list = [n for n in noun_list if len(n) > 1]
    t = ' '.join(noun_list)
    detokenized_doc.append(t)
    
df.txt = detokenized_doc 

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:11: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  # This is added back by InteractiveShellApp.init_path()


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features= 1000, # 상위 1,000개의 단어를 보존 
                             max_df = 0.5, smooth_idf=True)

X = vectorizer.fit_transform(df.txt)
X.shape # TF-IDF 행렬의 크기 확인

(5355, 1000)

In [ ]:
from sklearn.decomposition import TruncatedSVD
svd_model = TruncatedSVD(n_components=5, algorithm='randomized', n_iter=100, random_state=122)
svd_model.fit(X)
len(svd_model.components_)

np.shape(svd_model.components_)

terms = vectorizer.get_feature_names() # 단어 집합. 1,000개의 단어가 저장됨.

def get_topics(components, feature_names, n=5):
    for idx, topic in enumerate(components):
        print("Topic %d:" % (idx+1), [(feature_names[i], topic[i].round(5)) for i in topic.argsort()[:-n - 1:-1]])
get_topics(svd_model.components_,terms)

Topic 1: [('리뷰', 1.0), ('방문', 0.00051), ('기회', 0.00048), ('작품', 0.00037), ('기념관', 0.00022)]
Topic 2: [('방문', 0.47363), ('아이', 0.28716), ('산책', 0.25576), ('박물관', 0.21563), ('생각', 0.16199)]
Topic 3: [('방문', 0.83564), ('한번', 0.03036), ('다시', 0.02421), ('가치', 0.01604), ('필수', 0.01226)]
Topic 4: [('박물관', 0.74938), ('아이', 0.31648), ('역사', 0.07618), ('전시', 0.07608), ('체험', 0.06604)]
Topic 5: [('바다', 0.8288), ('박물관', 0.12844), ('해수욕장', 0.06743), ('정말', 0.06693), ('해변', 0.06492)]


/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)


In [ ]:
from sklearn.decomposition import TruncatedSVD
svd_model = TruncatedSVD(n_components=5, algorithm='randomized', n_iter=100, random_state=122)
svd_model.fit(X)
len(svd_model.components_)

np.shape(svd_model.components_)

terms = vectorizer.get_feature_names() # 단어 집합. 1,000개의 단어가 저장됨.

def get_topics(components, feature_names, n=7):
    for idx, topic in enumerate(components):
        print("Topic %d:" % (idx+1), [(feature_names[i], topic[i].round(5)) for i in topic.argsort()[:-n - 1:-1]])
get_topics(svd_model.components_,terms)

Topic 1: [('리뷰', 1.0), ('방문', 0.00051), ('기회', 0.00048), ('작품', 0.00037), ('기념관', 0.00022), ('소개', 0.00021), ('대한', 0.00019)]
Topic 2: [('방문', 0.47363), ('아이', 0.28716), ('산책', 0.25576), ('박물관', 0.21563), ('생각', 0.16199), ('공원', 0.15061), ('가족', 0.14992)]
Topic 3: [('방문', 0.83564), ('한번', 0.03036), ('다시', 0.02421), ('가치', 0.01604), ('필수', 0.01226), ('겨울', 0.01166), ('대구', 0.01002)]
Topic 4: [('박물관', 0.74938), ('아이', 0.31648), ('역사', 0.07618), ('전시', 0.07608), ('체험', 0.06604), ('국립', 0.03595), ('관람', 0.03086)]
Topic 5: [('바다', 0.8288), ('박물관', 0.12844), ('해수욕장', 0.06743), ('정말', 0.06693), ('해변', 0.06492), ('동해', 0.06393), ('겨울', 0.0488)]


/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)


In [ ]:
df = df.dropna(how = 'any')
df['리뷰내용_1'] = df['리뷰내용_1'].str.replace("[^ㄱ-ㅎㅏ-ㅣ가-힣 ]","")

okt = Okt()
tokenized_data = []

for sentence in df['리뷰내용_1']:
    temp_X = okt.nouns(sentence)
    temp_X = [word for word in temp_X if len(word)>1]
    tokenized_data.append(temp_X)

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:2: FutureWarning: The default value of regex will change from True to False in a future version.
  


In [ ]:
from gensim import corpora
dictionary = corpora.Dictionary(tokenized_doc)
corpus = [dictionary.doc2bow(text) for text in tokenized_doc]

In [ ]:
import gensim
NUM_TOPICS = 11
ldamodel = gensim.models.ldamodel.LdaModel(corpus, num_topics = NUM_TOPICS, id2word=dictionary, passes=15)
topics = ldamodel.print_topics(num_words=4)
for topic in topics:
    print(topic)

(0, '0.005*"있는" + 0.003*"있고" + 0.003*"있습니다" + 0.003*"너무"')
(1, '0.007*"있는" + 0.006*"있습니다" + 0.004*"너무" + 0.003*"있어"')
(2, '0.008*"있는" + 0.006*"박물관" + 0.005*"있습니다" + 0.005*"너무"')
(3, '0.009*"있는" + 0.003*"많이" + 0.003*"합니다" + 0.003*"좋은"')
(4, '0.003*"있는" + 0.003*"많이" + 0.003*"매우" + 0.003*"정말"')
(5, '0.007*"있는" + 0.005*"좋았습니다" + 0.004*"아름다운" + 0.003*"있어"')
(6, '0.007*"있는" + 0.006*"함께" + 0.004*"너무" + 0.004*"많이"')
(7, '0.010*"좋아요" + 0.008*"있는" + 0.005*"많이" + 0.005*"좋은"')
(8, '0.009*"좋은" + 0.005*"너무" + 0.004*"산책하기" + 0.004*"정말"')
(9, '0.011*"있는" + 0.005*"곳입니다" + 0.004*"좋습니다" + 0.003*"있어서"')
(10, '0.121*"리뷰없음" + 0.009*"좋은" + 0.006*"있는" + 0.003*"있어서"')


In [ ]:
def make_topictable_per_doc(ldamodel, corpus):
    topic_table = pd.DataFrame()

    # 몇 번째 문서인지를 의미하는 문서 번호와 해당 문서의 토픽 비중을 한 줄씩 꺼내온다.
    for i, topic_list in enumerate(ldamodel[corpus]):
        doc = topic_list[0] if ldamodel.per_word_topics else topic_list            
        doc = sorted(doc, key=lambda x: (x[1]), reverse=True)
        # 각 문서에 대해서 비중이 높은 토픽순으로 토픽을 정렬한다.
        # EX) 정렬 전 0번 문서 : (2번 토픽, 48.5%), (8번 토픽, 25%), (10번 토픽, 5%), (12번 토픽, 21.5%), 
        # Ex) 정렬 후 0번 문서 : (2번 토픽, 48.5%), (8번 토픽, 25%), (12번 토픽, 21.5%), (10번 토픽, 5%)
        # 48 > 25 > 21 > 5 순으로 정렬이 된 것.

        # 모든 문서에 대해서 각각 아래를 수행
        for j, (topic_num, prop_topic) in enumerate(doc): #  몇 번 토픽인지와 비중을 나눠서 저장한다.
            if j == 0:  # 정렬을 한 상태이므로 가장 앞에 있는 것이 가장 비중이 높은 토픽
                topic_table = topic_table.append(pd.Series([int(topic_num), round(prop_topic,4), topic_list]), ignore_index=True)
                # 가장 비중이 높은 토픽과, 가장 비중이 높은 토픽의 비중과, 전체 토픽의 비중을 저장한다.
            else:
                break
    return(topic_table)

In [ ]:
topictable = make_topictable_per_doc(ldamodel, corpus)
topictable = topictable.reset_index() # 문서 번호을 의미하는 열(column)로 사용하기 위해서 인덱스 열을 하나 더 만든다.
topictable.columns = ['문서 번호', '가장 비중이 높은 토픽', '가장 높은 토픽의 비중', '각 토픽의 비중']
topictable[:11]

,문서 번호,가장 비중이 높은 토픽,가장 높은 토픽의 비중,각 토픽의 비중
0,0,6.0,0.9526,"[(6, 0.952626)]"
1,1,1.0,0.9591,"[(1, 0.9590864)]"
2,2,2.0,0.9437,"[(2, 0.9437376)]"
3,3,6.0,0.9182,"[(6, 0.91817605)]"
4,4,2.0,0.9571,"[(2, 0.9571375)]"
5,5,2.0,0.9750,"[(2, 0.9749974)]"
6,6,1.0,0.8507,"[(1, 0.8507309), (4, 0.107158735)]"
7,7,1.0,0.9791,"[(1, 0.9790682)]"
8,8,8.0,0.9640,"[(8, 0.96399474)]"
9,9,3.0,0.9500,"[(3, 0.9499769)]"
